# Mascaramento de Dados Sensíveis nas Respostas de Ferramentas do Amazon Bedrock AgentCore Gateway

## Visão Geral

Este notebook demonstra como **anonimizar automaticamente Informações de Identificação Pessoal (PII)** nas respostas de ferramentas usando **interceptors do Amazon Bedrock AgentCore Gateway** integrados com **Amazon Bedrock Guardrails**. O interceptor inspeciona as respostas das ferramentas em tempo real e anonimiza dados sensíveis usando os recursos integrados de detecção e anonimização de PII do Bedrock antes de retornar os resultados aos clientes, garantindo conformidade com regulamentações de privacidade de dados.

### Por que Mascarar Dados Sensíveis no Gateway?

Ao construir aplicações de IA que acessam dados de clientes, você precisa proteger informações sensíveis:

- **Conformidade**: Atender aos requisitos do GDPR, HIPAA, PCI-DSS e outras regulamentações
- **Minimização de Dados**: Expor apenas as informações necessárias aos clientes
- **Proteção Centralizada**: Aplicar regras de anonimização de forma consistente em todas as ferramentas
- **Zero Trust**: Não depender de sistemas downstream para proteger dados sensíveis
- **Detecção Potencializada por IA**: Aproveitar a detecção avançada de PII do Bedrock Guardrails em mais de 31 tipos de PII

O interceptor do Gateway fornece um **ponto de aplicação centralizado** que anonimiza PII independentemente de qual ferramenta o retorna, sem modificar implementações individuais de ferramentas.

---

## O que Este Tutorial Aborda

Este tutorial implementa anonimização de PII usando um **interceptor RESPONSE** com **Amazon Bedrock Guardrails**:

🔒 **Anonimização de PII (interceptor RESPONSE + Bedrock Guardrails)**  
   - Cria um Bedrock Guardrail configurado com mais de 31 tipos de PII para detecção abrangente
   - Intercepta respostas de ferramentas do Gateway
   - Aplica Bedrock Guardrails para detectar e anonimizar dados sensíveis (emails, números de telefone, SSNs, cartões de crédito, endereços e mais)
   - Substitui PII detectada por placeholders anonimizados (ex.: , )
   - Retorna a resposta sanitizada ao cliente

![Arquitetura de Mascaramento de PII](images/PII-mask.png)

---

## Por que Usar Gateway Interceptors?

Gateway Interceptors permitem que você:

- **Proteção de Dados**: Anonimizar automaticamente informações sensíveis das respostas usando detecção potencializada por IA
- **Aplicação de Conformidade**: Aplicar políticas consistentes de proteção de dados em todas as ferramentas
- **Cobertura Abrangente**: Detectar mais de 31 tipos de PII incluindo nomes, endereços, dados financeiros, informações de saúde e mais
- **Auditoria & Governança**: Registrar eventos de acesso a dados e anonimização
- **Transformação de Respostas**: Modificar dados em trânsito sem alterar ferramentas
- **Serviço Gerenciado**: Aproveitar os modelos de detecção de PII continuamente atualizados do Bedrock Guardrails

Como os interceptors são acoplados na **camada do Gateway**, eles protegem dados de **qualquer** MCP server ou ferramenta subjacente sem modificar o código da aplicação.

---

## Detalhes do Tutorial

| Informação               | Detalhes                                                                     |
|--------------------------|------------------------------------------------------------------------------|
| **Tipo de tutorial**     | Interativo                                                                   |
| **Componentes AgentCore**| Amazon Bedrock AgentCore Gateway, Gateway Interceptors, Bedrock Guardrails  |
| **Tipo de target do Gateway** | MCP Server (ferramenta baseada em Lambda)                              |
| **Tipos de interceptor** | AWS Lambda (RESPONSE)                                                       |
| **Inbound Auth IdP**| Amazon Cognito (autorizador CUSTOM_JWT)                                    |
| **Proteção de Dados**    | Anonimização de PII usando Amazon Bedrock Guardrails (mais de 31 tipos de PII) |
| **Componentes do tutorial** | Gateway, Lambda Interceptor, Bedrock Guardrails, Amazon Cognito, MCP tools |
| **Vertical do tutorial** | Cross-vertical (aplicável a qualquer indústria com PII)                     |
| **Complexidade do exemplo** | Intermediário                                                             |
| **SDK utilizado**        | boto3                                                                        |

---

## Pré-requisitos

Para executar este tutorial você precisará de:

- Jupyter notebook (kernel Python)
- Credenciais AWS com permissões para:
  - AWS Lambda
  - AWS IAM
  - Amazon Cognito
  - Serviços Amazon Bedrock AgentCore (control plane)
  - Amazon Bedrock Guardrails (bedrock:CreateGuardrail, bedrock:ApplyGuardrail)
- Python 3.9 ou superior
- Compreensão básica de AWS Lambda, IAM roles, Amazon Cognito, Amazon Bedrock Guardrails e Amazon Bedrock AgentCore Gateway

> ⚠️ **Nota:** A seção de Limpeza no final exclui os recursos AWS criados por este tutorial (Gateway, Lambdas, IAM roles, Guardrails, etc.). Execute-a apenas quando estiver pronto para remover tudo.


---

## Parte 1: Configuração & Implantação

### Passo 1.0: Instalar Dependências Necessárias

Instale todos os pacotes Python necessários para este tutorial.

In [ ]:
!pip install -r requirements.txt

### Passo 1.1: Importar Bibliotecas Necessárias

In [ ]:
import boto3
import json
import time
import sys
from pathlib import Path
from datetime import datetime
from botocore.exceptions import ClientError

# Add parent directory to path for utils
utils_dir = Path.cwd().parent
sys.path.insert(0, str(utils_dir))

import utils

print("✓ Libraries imported")

# Generate unique identifier for this deployment
DEPLOYMENT_ID = datetime.now().strftime('%Y%m%d-%H%M%S')
print(f"\nDeployment ID: {DEPLOYMENT_ID}")

### Passo 1.2: Configurar Variáveis de Implantação

In [ ]:
# Configuration
REGION = "us-east-1"  
LAMBDA_FUNCTION_NAME = f"interceptor-lambda-{DEPLOYMENT_ID}"
LAMBDA_ROLE_NAME = f"interceptor-lambda-role-{DEPLOYMENT_ID}"
GATEWAY_NAME = f"interceptor-gateway-{DEPLOYMENT_ID}"

# Initialize clients
gateway_client = boto3.client('bedrock-agentcore-control', region_name=REGION)
cognito_client = boto3.client('cognito-idp', region_name=REGION)

print("Configuration:")
print(f"  Lambda Function: {LAMBDA_FUNCTION_NAME}")
print(f"  Lambda Role: {LAMBDA_ROLE_NAME}")
print(f"  Gateway Name: {GATEWAY_NAME}")
print(f"  Region: {REGION}")


### Passo 1.3: Configurar Bedrock Guardrails para Filtragem de Dados Sensíveis

Crie um Bedrock Guardrail com filtros de dados sensíveis para anonimizar PII usando os recursos integrados do Amazon Bedrock.

In [ ]:
# Setup Bedrock Guardrails for sensitive data filtering
print("Creating Bedrock Guardrail with sensitive data filters...")

bedrock_client = boto3.client('bedrock', region_name=REGION)

GUARDRAIL_NAME = f"pii-masking-guardrail-{DEPLOYMENT_ID}"

# Define sensitive data filters with ANONYMIZE behavior
# Bedrock Guardrails supports 31 PII entity types by default
# ANONYMIZE action replaces PII with placeholder text (masking behavior)
sensitive_information_policy_config = {
    'piiEntitiesConfig': [
        # General PII
        {'type': 'ADDRESS', 'action': 'ANONYMIZE'},
        {'type': 'AGE', 'action': 'ANONYMIZE'},
        {'type': 'EMAIL', 'action': 'ANONYMIZE'},
        {'type': 'NAME', 'action': 'ANONYMIZE'},
        {'type': 'PHONE', 'action': 'ANONYMIZE'},
        {'type': 'USERNAME', 'action': 'ANONYMIZE'},
        {'type': 'PASSWORD', 'action': 'ANONYMIZE'},
        
        # Financial Information
        {'type': 'CREDIT_DEBIT_CARD_CVV', 'action': 'ANONYMIZE'},
        {'type': 'CREDIT_DEBIT_CARD_EXPIRY', 'action': 'ANONYMIZE'},
        {'type': 'CREDIT_DEBIT_CARD_NUMBER', 'action': 'ANONYMIZE'},
        {'type': 'PIN', 'action': 'ANONYMIZE'},
        {'type': 'INTERNATIONAL_BANK_ACCOUNT_NUMBER', 'action': 'ANONYMIZE'},
        {'type': 'SWIFT_CODE', 'action': 'ANONYMIZE'},
        
        # US-Specific Identifiers
        {'type': 'US_BANK_ACCOUNT_NUMBER', 'action': 'ANONYMIZE'},
        {'type': 'US_BANK_ROUTING_NUMBER', 'action': 'ANONYMIZE'},
        {'type': 'US_INDIVIDUAL_TAX_IDENTIFICATION_NUMBER', 'action': 'ANONYMIZE'},
        {'type': 'US_PASSPORT_NUMBER', 'action': 'ANONYMIZE'},
        {'type': 'US_SOCIAL_SECURITY_NUMBER', 'action': 'ANONYMIZE'},
        {'type': 'DRIVER_ID', 'action': 'ANONYMIZE'},
        
        # UK-Specific Identifiers
        {'type': 'UK_NATIONAL_HEALTH_SERVICE_NUMBER', 'action': 'ANONYMIZE'},
        {'type': 'UK_NATIONAL_INSURANCE_NUMBER', 'action': 'ANONYMIZE'},
        {'type': 'UK_UNIQUE_TAXPAYER_REFERENCE_NUMBER', 'action': 'ANONYMIZE'},
        
        # Canada-Specific Identifiers
        {'type': 'CA_HEALTH_NUMBER', 'action': 'ANONYMIZE'},
        {'type': 'CA_SOCIAL_INSURANCE_NUMBER', 'action': 'ANONYMIZE'},
        
        # Network & Technical
        {'type': 'IP_ADDRESS', 'action': 'ANONYMIZE'},
        {'type': 'MAC_ADDRESS', 'action': 'ANONYMIZE'},
        {'type': 'URL', 'action': 'ANONYMIZE'},
        
        # AWS Credentials
        {'type': 'AWS_ACCESS_KEY', 'action': 'ANONYMIZE'},
        {'type': 'AWS_SECRET_KEY', 'action': 'ANONYMIZE'},
        
        # Vehicle Identification
        {'type': 'VEHICLE_IDENTIFICATION_NUMBER', 'action': 'ANONYMIZE'},
        {'type': 'LICENSE_PLATE', 'action': 'ANONYMIZE'}
    ]
}

try:
    # Create the guardrail
    guardrail_response = bedrock_client.create_guardrail(
        name=GUARDRAIL_NAME,
        description='Guardrail for anonymizing sensitive PII data in tool responses',
        sensitiveInformationPolicyConfig=sensitive_information_policy_config,
        blockedInputMessaging='Input contains sensitive information that has been anonymized.',
        blockedOutputsMessaging='Output contains sensitive information that has been anonymized.'
    )
    
    GUARDRAIL_ID = guardrail_response['guardrailId']
    GUARDRAIL_ARN = guardrail_response['guardrailArn']
    
    print(f"✓ Guardrail created: {GUARDRAIL_ID}")
    print(f"  ARN: {GUARDRAIL_ARN}")
    
    # Create a version of the guardrail
    print("\nCreating guardrail version...")
    version_response = bedrock_client.create_guardrail_version(
        guardrailIdentifier=GUARDRAIL_ID,
        description='Initial version with PII anonymization'
    )
    
    GUARDRAIL_VERSION = version_response['version']
    print(f"✓ Guardrail version created: {GUARDRAIL_VERSION}")
    
    # Display configured PII types
    print(f"\n✓ Configured {len(sensitive_information_policy_config['piiEntitiesConfig'])} PII types for anonymization:")
    for pii_config in sensitive_information_policy_config['piiEntitiesConfig']:
        print(f"  - {pii_config['type']}: {pii_config['action']}")
    
except ClientError as e:
    error_code = e.response['Error']['Code']
    if error_code == 'ConflictException':
        print(f"⚠ Guardrail with name '{GUARDRAIL_NAME}' already exists")
        # List existing guardrails to get the ID
        list_response = bedrock_client.list_guardrails()
        for guardrail in list_response.get('guardrails', []):
            if guardrail['name'] == GUARDRAIL_NAME:
                GUARDRAIL_ID = guardrail['id']
                GUARDRAIL_ARN = guardrail['arn']
                print(f"  Using existing guardrail: {GUARDRAIL_ID}")
                
                # Get the latest version of the existing guardrail
                try:
                    get_response = bedrock_client.get_guardrail(
                        guardrailIdentifier=GUARDRAIL_ID
                    )
                    GUARDRAIL_VERSION = get_response.get('version', 'DRAFT')
                    print(f"  Guardrail version: {GUARDRAIL_VERSION}")
                except Exception as get_error:
                    print(f"  ⚠ Could not get guardrail version: {get_error}")
                    GUARDRAIL_VERSION = 'DRAFT'
                break
    else:
        print(f"✗ Failed to create guardrail: {e}")
        raise
except Exception as e:
    print(f"✗ Unexpected error creating guardrail: {e}")
    raise


### Passo 1.4: Criar IAM Role para o Lambda Interceptor

Conceda permissões ao Lambda para execução e escrita de logs no CloudWatch.

In [ ]:
# Create IAM role for Lambda interceptor using utils
print("Creating IAM role for Lambda interceptor...")

LAMBDA_ROLE_ARN = utils.create_lambda_role(
    role_name=LAMBDA_ROLE_NAME,
    description='Role for AgentCore Lambda Interceptor for PII masking with Bedrock Guardrails'
)

print(f"  ARN: {LAMBDA_ROLE_ARN}")

# Add Bedrock Guardrails permissions to Lambda role
print("\nAdding Bedrock Guardrails permissions to Lambda role...")
iam_client = boto3.client('iam')

bedrock_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": [
                "bedrock:ApplyGuardrail"
            ],
            "Resource": "*"
        }
    ]
}

try:
    iam_client.put_role_policy(
        RoleName=LAMBDA_ROLE_NAME,
        PolicyName='BedrockGuardrailsPolicy',
        PolicyDocument=json.dumps(bedrock_policy)
    )
    print("✓ Bedrock Guardrails permissions added")
    print("  Policy: bedrock:ApplyGuardrail on all resources")
except Exception as e:
    print(f"⚠ Failed to add Bedrock permissions: {e}")

### Passo 1.4a: Aguardar o Bedrock Guardrail Ficar Pronto

Aguarde tempo para o Bedrock Guardrail se propagar e ficar totalmente disponível.

In [ ]:
time.sleep(10)

### Passo 1.5: Implantar Função Lambda Interceptor

O Lambda intercepta respostas de ferramentas e mascara PII usando Bedrock Guardrails.

In [ ]:
# Deploy Lambda interceptor using utils
print("Deploying Lambda interceptor...")

# Prepare environment variables for Lambda
lambda_env_vars = {}
if 'GUARDRAIL_ID' in globals():
    lambda_env_vars['GUARDRAIL_ID'] = GUARDRAIL_ID
    lambda_env_vars['GUARDRAIL_VERSION'] = GUARDRAIL_VERSION
    print(f"  Configuring Lambda with Guardrail: {GUARDRAIL_ID} (version: {GUARDRAIL_VERSION})")
else:
    print("  ⚠ WARNING: GUARDRAIL_ID not found. Lambda will skip PII masking.")
    print("  Make sure to run Step 1.2a to create the Guardrail first.")

LAMBDA_ARN = utils.deploy_lambda_function(
    function_name=LAMBDA_FUNCTION_NAME,
    role_arn=LAMBDA_ROLE_ARN,
    lambda_code_path='src/lambda/lambda_function.py',
    description='AgentCore Response Lambda Interceptor to mask sensitive data using Bedrock Guardrails',
    timeout=30,
    memory_size=256,
    environment_vars=lambda_env_vars if lambda_env_vars else None,
    region=REGION
)

print(f"  ARN: {LAMBDA_ARN}")

### Passo 1.5a: Conceder Permissão ao Gateway para Invocar o Lambda

Adicione permissões para o Gateway invocar a função Lambda interceptor.

In [ ]:
# Grant Gateway permission to invoke the Lambda interceptor
print("\nGranting Gateway permission to invoke Lambda...")

utils.grant_gateway_invoke_permission(
    function_name=LAMBDA_FUNCTION_NAME,
    region=REGION
)

### Passo 1.6: Criar Amazon Cognito User Pool & App Client

Crie um Cognito user pool para autenticação do Gateway usando o fluxo OAuth client credentials.

In [ ]:
# Create Cognito User Pool and Client for Gateway authentication using utils
print("Creating Cognito User Pool and Client...")

USER_POOL_NAME = f"gateway-pool-{DEPLOYMENT_ID}"
RESOURCE_SERVER_ID = 'gateway'
RESOURCE_SERVER_NAME = 'Gateway Resource Server'
SCOPES = [{'ScopeName': 'tools', 'ScopeDescription': 'Access to gateway tools'}]

# Create or get user pool
USER_POOL_ID = utils.get_or_create_user_pool(cognito_client, USER_POOL_NAME)
print(f"  Pool ID: {USER_POOL_ID}")

# Create or get resource server
utils.get_or_create_resource_server(cognito_client, USER_POOL_ID, RESOURCE_SERVER_ID, RESOURCE_SERVER_NAME, SCOPES)

# Wait for resource server to propagate
print("  Waiting for resource server to propagate...")
time.sleep(3)

# Create M2M client with client credentials flow
CLIENT_NAME = f"gateway-client-{DEPLOYMENT_ID}"
CLIENT_ID, CLIENT_SECRET = utils.get_or_create_m2m_client(
    cognito_client,
    USER_POOL_ID,
    CLIENT_NAME,
    RESOURCE_SERVER_ID,
    SCOPES=[f"{RESOURCE_SERVER_ID}/tools"]
)

print(f"✓ User Pool Client created: {CLIENT_NAME}")
print(f"  Client ID: {CLIENT_ID}")
print(f"  Client Secret: {CLIENT_SECRET[:20]}...")

# Construct OAuth URLs
POOL_DOMAIN = USER_POOL_ID.replace('_', '').lower()
COGNITO_DOMAIN = f"https://{POOL_DOMAIN}.auth.{REGION}.amazoncognito.com"
DISCOVERY_URL = f"https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration"
TOKEN_URL = f"{COGNITO_DOMAIN}/oauth2/token"

print(f"\n✓ OAuth Configuration:")
print(f"  Discovery URL: {DISCOVERY_URL}")
print(f"  Token URL: {TOKEN_URL}")
print(f"  Scope: {RESOURCE_SERVER_ID}/tools")

### Passo 1.7: Criar Gateway com Response Interceptor

**Por que o Interceptor RESPONSE?**  
O interceptor processa respostas de ferramentas após a execução, permitindo mascarar PII antes de retornar dados aos clientes.

In [ ]:
# Create Gateway IAM role
gateway_iam_role = utils.create_agentcore_gateway_role_with_region(GATEWAY_NAME, REGION)
GATEWAY_ROLE_ARN = gateway_iam_role['Role']['Arn']

print(f"✓ Gateway role created: {GATEWAY_ROLE_ARN}")

# Wait for role propagation
time.sleep(10)

# Create Gateway with Lambda interceptor
print(f"\nCreating Gateway with RESPONSE interceptor...")

try:
    gateway_response = gateway_client.create_gateway(
        name=GATEWAY_NAME,
        protocolType="MCP",
        protocolConfiguration={
            "mcp": {
                "supportedVersions": ["2025-03-26"]
            }
        },
        interceptorConfigurations=[
            {
                "interceptor": {
                    "lambda": {
                        "arn": LAMBDA_ARN
                    }
                },
                "interceptionPoints": ["RESPONSE"],
                "inputConfiguration": {
                    "passRequestHeaders": True  
                }
            }
        ],
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": DISCOVERY_URL,
                "allowedClients": [CLIENT_ID]
            }
        },
        roleArn=GATEWAY_ROLE_ARN
    )
    
    GATEWAY_ID = gateway_response.get('gatewayId')
    print(f"✓ Gateway created: {GATEWAY_ID}")
    
except Exception as e:
    print(f"\n✗ Failed to create Gateway: {e}")
    raise


### Passo 1.8: Aguardar o Gateway Ficar Pronto

In [ ]:
# Wait for Gateway to be ready using signed requests
print("\nWaiting for Gateway to be ready...")

max_attempts = 30
for attempt in range(max_attempts):
    try:
        response = gateway_client.get_gateway(gatewayIdentifier=GATEWAY_ID)
        status_code = response.get("ResponseMetadata", {}).get("HTTPStatusCode")

        if status_code == 200:
            # gateway_info = response.json()
            status = response.get('status', 'UNKNOWN')
            
            print(f"  [{attempt + 1}/{max_attempts}] Status: {status}")
            
            if status == 'READY':
                GATEWAY_URL = response.get('gatewayUrl')
                print(f"\n✓ Gateway is ready!")
                print(f"  URL: {GATEWAY_URL}")
                
                # Show interceptor configuration
                if 'interceptorConfigurations' in response:
                    interceptor_configs = response['interceptorConfigurations']
                    print(f"\n  Interceptor Configuration:")
                    for idx, config in enumerate(interceptor_configs):
                        print(f"    [{idx}] Interception Points: {config.get('interceptionPoints', [])}")
                        print(f"    [{idx}] Lambda ARN: {config.get('interceptor', {}).get('lambda', {}).get('arn', 'N/A')}")
                        print(f"    [{idx}] Pass Headers: {config.get('inputConfiguration', {}).get('passRequestHeaders', False)}")
                break
            elif status == 'FAILED':
                print(f"\n✗ Gateway creation failed")
                print(f"  Details: {response}")
                raise Exception("Gateway failed")
        else:
            print(f"  [{attempt + 1}/{max_attempts}] HTTP Error: {response.status_code}")
    except Exception as e:
        print(f"  [{attempt + 1}/{max_attempts}] Error: {e}")
    
    time.sleep(10)
else:
    print(f"\n⚠ Timeout waiting for Gateway")
    raise Exception("Gateway timeout")


### Passo 1.9: Registrar Ferramentas de Exemplo no Gateway

Implante o Lambda de ferramenta de exemplo (dados de funcionário) e registre-o como target do Gateway.

In [ ]:
# Deploy tool Lambdas and register as Gateway targets
print("Deploying tool Lambda functions...")

# Import tool modules
sys.path.insert(0, str(Path.cwd()))
from src.tools import employee_data_tool

# Create IAM role for tool Lambdas using utils
TOOL_ROLE_ARN = utils.create_lambda_role(
    role_name=f"tool-lambda-role-{DEPLOYMENT_ID}",
    description='Role for tool Lambda functions'
)

# Deploy tool Lambda functions
tools_to_deploy = [
    ('employee_data_tool', employee_data_tool),
]

deployed_tools = []

for tool_name, tool_module in tools_to_deploy:
    print(f"  Deploying {tool_name}...")
    
    function_name = f"{tool_name.replace('_', '-')}-{DEPLOYMENT_ID}"
    tool_code_path = Path(tool_module.__file__)
    
    lambda_arn = utils.deploy_lambda_function(
        function_name=function_name,
        role_arn=TOOL_ROLE_ARN,
        lambda_code_path=str(tool_code_path),
        environment_vars={'TOOL_NAME': tool_name},
        description=f'{tool_name} function',
        region=REGION
    )
    
    tool_definition = getattr(tool_module, 'TOOL_DEFINITION', {
        "name": tool_name,
        "description": f"{tool_name} function"
    })
    
    deployed_tools.append({
        'tool_name': tool_name,
        'function_name': function_name,
        'lambda_arn': lambda_arn,
        'tool_definition': tool_definition
    })

print(f"✓ Deployed {len(deployed_tools)} tool Lambdas")

# Register tools as Gateway targets
print("\nRegistering tools as Gateway targets...")
created_targets = []

for tool in deployed_tools:
    print(f"  Registering {tool['tool_name']}...")
    
    try:
        response = gateway_client.create_gateway_target(
            gatewayIdentifier=GATEWAY_ID,
            name=f"{tool['tool_name'].replace('_', '-')}-target",
            targetConfiguration={
                "mcp": {
                    "lambda": {
                        "lambdaArn": tool["lambda_arn"],
                        "toolSchema": {"inlinePayload": [tool["tool_definition"]]}
                    }
                }
            },
            credentialProviderConfigurations=[{
                "credentialProviderType": "GATEWAY_IAM_ROLE"
            }]
        )
        
        target_id = response['targetId']
        print(f"    ✓ Target created: {target_id}")
        
        # Wait for target to be READY
        for attempt in range(18):
            status_response = gateway_client.get_gateway_target(
                gatewayIdentifier=GATEWAY_ID,
                targetId=target_id
            )
            status = status_response.get('status')
            
            if status == 'READY':
                print(f"    ✓ Target is READY")
                created_targets.append({
                    'tool_name': tool['tool_name'],
                    'target_id': target_id,
                    'lambda_arn': tool['lambda_arn']
                })
                break
            elif status == 'FAILED':
                print(f"    ✗ Target FAILED")
                break
            
            time.sleep(10)
            
    except Exception as e:
        print(f"    ✗ Failed to create target: {e}")

# Summary
print(f"\n✓ Deployed {len(deployed_tools)} tool Lambdas")
print(f"✓ Created {len(created_targets)} gateway targets")

if len(created_targets) < len(deployed_tools):
    print(f"⚠ Warning: Not all targets were created successfully")

# Store for cleanup
DEPLOYED_TOOL_FUNCTIONS = [t['function_name'] for t in deployed_tools]
CREATED_TARGET_IDS = [t['target_id'] for t in created_targets]


---

## Parte 2: Testes

### Passo 2.1: Testar Anonimização de PII com Bedrock Guardrails

Invoque a ferramenta de dados de funcionário e verifique se a PII é anonimizada na resposta.

#### O que Esperar:

A ferramenta de dados de funcionário retorna informações realistas de funcionários contendo vários tipos de PII. O Lambda interceptor irá:

1. **Interceptar a resposta** após a execução da ferramenta
2. **Aplicar Bedrock Guardrails** para detectar PII em mais de 31 tipos de entidade
3. **Anonimizar PII detectada** substituindo-a por tokens de placeholder
4. **Retornar a resposta sanitizada** ao cliente

#### Formato de Anonimização do Bedrock Guardrails:

Bedrock Guardrails substitui PII detectada por placeholders anonimizados no formato:

- **Emails**:  → 
- **Números de Telefone**:  → 
- **Nomes**:  → 
- **Endereços**:  → 
- **SSN**:  → 
- **Cartões de Crédito**:  → 
- **Endereços IP**:  → 
- **URLs**:  → 

#### Exemplo de Saída:

**Antes da anonimização (resposta bruta da ferramenta):**


**Após a anonimização (resposta interceptada):**


Observe que dados não sensíveis como ,  e  permanecem inalterados, enquanto toda PII (email, endereço) é substituída por placeholders anonimizados.



In [ ]:
# Test the PII masking by invoking the tool
import requests

print("Testing PII masking interceptor...")
print(f"Gateway URL: {GATEWAY_URL}")

# Get OAuth token
token_data = utils.get_token(
    user_pool_id=USER_POOL_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    scope_string="gateway/tools",
    REGION=REGION
)

if 'error' in token_data:
    print(f"✗ Token request failed: {token_data['error']}")
else:
    token = token_data['access_token']
    print(f"✓ Token obtained")

### Passo 2.2: Testar Ferramenta de Dados de Funcionário

Invoque a ferramenta de dados de funcionário para ver como o Bedrock Guardrails anonimiza PII abrangente incluindo informações de contato e dados financeiros, mesmo quando os nomes dos campos não indicam sensibilidade explicitamente.

A ferramenta de dados de funcionário retorna:
- **Informações de Contato**: Email e endereço físico
- **Informações Financeiras**: Números de conta bancária, números de roteamento, detalhes de cartão de crédito, CVV, datas de validade, PIN e ID fiscal
- **Dados Não Sensíveis**: ID do funcionário, departamento, status, saldos de conta, pontuações de crédito

**Antes da anonimização:**


**Após a anonimização:**


**Observações Importantes:**
- **Detecção Baseada em Conteúdo**: Nomes de campo como  e  não dizem explicitamente "email" ou "address", mas o Bedrock Guardrails detecta e anonimiza o conteúdo com base no reconhecimento de padrões
- **Proteção Abrangente de PII Financeira**: Todos os dados financeiros sensíveis (contas bancárias, cartões de crédito, IDs fiscais) são automaticamente detectados e anonimizados
- **Anonimização Seletiva**: Dados financeiros não sensíveis como saldos de conta e pontuações de crédito permanecem inalterados
- **Mais de 31 Tipos de PII**: Bedrock Guardrails protege contra uma ampla gama de tipos de PII sem necessidade de correspondência explícita de nomes de campo

In [ ]:
# Test the employee data tool
print("\n" + "="*60)
print("Testing Employee Data Tool with PII Anonymization")
print("="*60)

# Reuse the token from previous step
if 'token' in locals():
    print(f"✓ Using existing token")
    
    # Call the employee data tool
    mcp_request = {
        "jsonrpc": "2.0",
        "method": "tools/call",
        "id": 2,
        "params": {
            "name": "employee-data-tool-target___employee_data_tool",
            "arguments": {"employee_id": "EMP-98765"}
        }
    }
    
    response = requests.post(
        GATEWAY_URL,
        headers={
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json"
        },
        json=mcp_request
    )
    
    if response.status_code == 200:
        result = response.json()
        print(f"\n✓ Employee tool invoked successfully")
        print(f"\nResponse (PII in contact_info and mailing_info should be anonymized):")
        print(json.dumps(result, indent=2))
        
        # Highlight the anonymization
        print(f"\n📝 Notice:")
        print(f"  - 'employee_id', 'department', and 'status' remain unchanged (non-sensitive)")
        print(f"  - 'contact_info' email is replaced with [EMAIL] placeholder")
        print(f"  - 'mailing_info' address is replaced with [ADDRESS] placeholder")
        print(f"  - All financial PII is anonymized:")
        print(f"    • Bank account → [US_BANK_ACCOUNT_NUMBER]")
        print(f"    • Routing number → [US_BANK_ROUTING_NUMBER]")
        print(f"    • Credit card → [CREDIT_DEBIT_CARD_NUMBER]")
        print(f"    • CVV → [CREDIT_DEBIT_CARD_CVV]")
        print(f"    • Card expiry → [CREDIT_DEBIT_CARD_EXPIRY]")
        print(f"    • PIN → [PIN]")
        print(f"    • Tax ID → [US_INDIVIDUAL_TAX_IDENTIFICATION_NUMBER]")
        print(f"  - Non-sensitive financial data preserved (balances, credit scores, currency)")
        print(f"\n  ⭐ Key Point: Bedrock Guardrails uses content-based detection, not field names.")
        print(f"  Field names like 'mailing_info' don't explicitly say 'address', but the content")
        print(f"  is still detected and anonymized. This works across 31+ PII types automatically!")
    else:
        print(f"✗ Request failed: {response.status_code}")
        print(f"Response: {response.text}")
else:
    print("✗ No token available. Please run Step 2.1 first.")

---

# Parte 3: Limpeza - Excluir Todos os Recursos

⚠️ **AVISO: Isso irá EXCLUIR todos os recursos criados na Parte 1!**

Execute esta seção apenas se quiser limpar tudo.

### Passo 3.1: Excluir Recursos Criados

In [ ]:
# Cleanup - Delete all created resources using utils
print("Starting cleanup...")

# 1. Delete gateway targets
if 'CREATED_TARGET_IDS' in globals() and 'GATEWAY_ID' in globals():
    utils.delete_gateway_targets(gateway_client, GATEWAY_ID, CREATED_TARGET_IDS)
    # Wait for target deletions to complete before deleting gateway
    time.sleep(5)

# 2. Delete gateway
if 'GATEWAY_ID' in globals():
    utils.delete_gateway(gateway_client, GATEWAY_ID)
    print("✓ Deleted gateway")

# 3. Delete Lambda functions (tools + interceptor)
lambda_functions_to_delete = []
if 'DEPLOYED_TOOL_FUNCTIONS' in globals():
    lambda_functions_to_delete.extend(DEPLOYED_TOOL_FUNCTIONS)
if 'LAMBDA_FUNCTION_NAME' in globals():
    lambda_functions_to_delete.append(LAMBDA_FUNCTION_NAME)

if lambda_functions_to_delete:
    utils.delete_lambda_functions(lambda_functions_to_delete, REGION)

# 4. Delete IAM roles
if 'LAMBDA_ROLE_NAME' in globals():
    utils.delete_iam_role(LAMBDA_ROLE_NAME)
if 'DEPLOYMENT_ID' in globals():
    utils.delete_iam_role(f"tool-lambda-role-{DEPLOYMENT_ID}")
    utils.delete_iam_role(f"agentcore-{GATEWAY_NAME}-role")

# 5. Delete Cognito user pool
if 'USER_POOL_ID' in globals():
    utils.delete_cognito_user_pool(USER_POOL_ID, REGION)

# 6. Delete Bedrock Guardrail
if 'GUARDRAIL_ID' in globals():
    try:
        print("\nDeleting Bedrock Guardrail...")
        bedrock_client.delete_guardrail(guardrailIdentifier=GUARDRAIL_ID)
        print(f"✓ Deleted guardrail: {GUARDRAIL_ID}")
    except Exception as e:
        print(f"⚠ Failed to delete guardrail: {e}")

print("\n✓ Cleanup complete!")

---

# Resumo

Este notebook demonstra o mascaramento de PII usando Lambda interceptors:

1. ✅ **Configuração** - Criou Lambda interceptor, IAM roles, Cognito e Gateway
2. ✅ **Teste** - Verificou o mascaramento de PII nas respostas do Gateway
3. ✅ **Limpeza** - Excluiu todos os recursos

## O que Demonstramos

- **Lambda RESPONSE interceptor** que mascara dados sensíveis nas respostas de ferramentas
- **Detecção e mascaramento de PII** usando padrões regex
- **Integração com Gateway** com interceptors personalizados
- **Gerenciamento completo do ciclo de vida** dos recursos

## Próximos Passos

- Personalize os padrões de mascaramento para seu caso de uso
- Adicione detecção de PII mais sofisticada (ex.: AWS Comprehend)
- Integre com logging de conformidade
- Monitore os logs do CloudWatch para depuração